# 06.4 — Data Cleaning

## Learning Objectives

By the end of this notebook you should be able to:

- Detect and count missing values with `.isnull()` and `.notna()`
- Handle missing values using `.dropna()` and `.fillna()`
- Find and remove duplicate rows
- Rename columns with `.rename()`
- Convert data types with `.astype()` and `pd.to_numeric()`
- Clean and transform string columns using the `.str` accessor

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
raw = pd.read_csv(url)
raw.columns = raw.columns.str.lower()
raw = raw.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
print(raw.shape)
raw.head()

## 1. Why Data Cleaning Matters

Real datasets are almost never clean. Missing values, wrong data types, inconsistent strings, and duplicate rows are all common. This dataset happens to be fairly tidy — we will use it to learn the tools, and we'll introduce artificial imperfections where needed to demonstrate key techniques.

A useful first step on any dataset: check for missing values.

In [ ]:
# How clean is the Titanic dataset?
raw.isnull().sum()

This dataset has no missing values — which is unusually clean. To demonstrate the missing-value tools properly, we'll create a realistic version with ~20% of ages randomly removed, as is common in real Titanic sources.

In [ ]:
# Create a version with missing ages for demonstration
rng = np.random.default_rng(42)
df = raw.copy()
missing_idx = rng.choice(df.index, size=int(0.2 * len(df)), replace=False)
df.loc[missing_idx, "age"] = np.nan

print("Missing values per column:")
print(df.isnull().sum())

## 2. Detecting Missing Values

`.isnull()` returns a boolean DataFrame — `True` wherever a value is missing. Chaining `.sum()` counts missing values per column.

In [ ]:
# Count and percentage missing
total = len(df)
missing = df.isnull().sum()
pct = (missing / total * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": pct}).query("missing_count > 0")

In [ ]:
# Which rows have at least one missing value?
rows_with_missing = df[df.isnull().any(axis=1)]
print(f"Rows with at least one missing value: {len(rows_with_missing)}")
rows_with_missing.head()

## 3. Dropping Missing Values with `.dropna()`

The simplest response to missing data is to drop affected rows or columns.

Key parameters:
- `axis=0` (default) — drop **rows**; `axis=1` — drop **columns**
- `how="any"` (default) — drop if *any* value is missing
- `how="all"` — drop only if *all* values are missing
- `subset=[...]` — only check specific columns
- `thresh=n` — keep rows with at least `n` non-null values

In [ ]:
df_dropped = df.dropna()
print(f"Original rows: {len(df)}")
print(f"After dropna: {len(df_dropped)}")

In [ ]:
# More targeted: drop only rows where 'age' is missing
df_with_age = df.dropna(subset=["age"])
print(f"Rows with age data: {len(df_with_age)}")

## 4. Filling Missing Values with `.fillna()`

Often it is better to fill missing values than to drop them. Common strategies:

| Strategy | When to use |
|---|---|
| Fill with a constant (`0`, `"Unknown"`) | Absence has meaning; categorical data |
| Fill with column median | Numeric; robust to outliers |
| Fill with column mean | Numeric; only when distribution is roughly symmetric |
| Forward fill (`ffill`) | Time-ordered data where the previous value is a good estimate |

In [ ]:
# Fill missing ages with the median
median_age = df["age"].median()
print(f"Median age: {median_age}")

df_filled = df.copy()
df_filled["age"] = df_filled["age"].fillna(median_age)
print(f"Missing ages after fill: {df_filled['age'].isnull().sum()}")

In [ ]:
# Forward fill — each missing value gets the value from the row above
df_ffill = df.copy()
df_ffill["age"] = df_ffill["age"].ffill()
print(f"Missing ages after ffill: {df_ffill['age'].isnull().sum()}")

> **Important:** In machine learning, never compute the fill value (e.g., median age) using the test set. Always compute it from the training data only, then apply the same value to the test set. This prevents data leakage.

## 5. Duplicate Rows

`.duplicated()` returns a boolean Series — `True` for rows that are exact duplicates of a previous row. `.drop_duplicates()` removes them.

In [ ]:
# The Titanic dataset has no duplicates
print("Duplicates in original:", raw.duplicated().sum())

# Create a version with duplicates to demonstrate
df_with_dupes = pd.concat([raw.head(5), raw.head(3)], ignore_index=True)
print(f"Rows with duplicates: {len(df_with_dupes)}")
print(f"Duplicate rows:       {df_with_dupes.duplicated().sum()}")

In [ ]:
df_deduped = df_with_dupes.drop_duplicates()
print(f"Rows after dedup: {len(df_deduped)}")

## 6. Renaming Columns

`.rename()` changes column names without modifying the data. Pass a dictionary mapping old names to new names.

In [ ]:
df_renamed = raw.rename(columns={
    "sibsp": "siblings_spouses",
    "parch": "parents_children",
    "pclass": "passenger_class",
})
print(df_renamed.columns.tolist())

## 7. Converting Data Types

`.astype()` changes a column's data type. The **category** type is efficient for columns with a small number of distinct values.

In [ ]:
# Current types
raw.dtypes

In [ ]:
df_typed = raw.copy()
df_typed["survived"] = df_typed["survived"].astype("category")
df_typed["pclass"]   = df_typed["pclass"].astype("category")
df_typed["sex"]      = df_typed["sex"].astype("category")
df_typed.dtypes

In [ ]:
# pd.to_numeric() is safer when a column might contain non-numeric strings
# errors='coerce' turns bad values into NaN rather than raising an error
sample = pd.Series(["22", "38", "unknown", "35", "28"])
pd.to_numeric(sample, errors="coerce")

## 8. Cleaning String Columns

The `.str` accessor handles common string cleaning operations.

In [ ]:
# Demonstrate on a messy version of the sex column
messy_sex = pd.Series(["  Male", "female", "MALE", "Female ", "male"])
clean_sex = messy_sex.str.strip().str.lower()
print(clean_sex.unique())

The `name` column contains the passenger's title (Mr., Mrs., Miss., Master., Dr., etc.) embedded in the full name. For example: `"Mr. Owen Harris Braund"`. We can extract just the title using `.str.extract()` with a regular expression.

In [ ]:
# Extract the title: first word ending in a period
# Names look like: "Mr. Owen Harris Braund"
raw["title"] = raw["name"].str.extract(r'^(\w+\.)')
raw["title"].value_counts()

## 9. A Complete Cleaning Workflow

Here is a realistic end-to-end workflow: start from the raw CSV, introduce missing age values (as they exist in other Titanic sources), and produce a clean, analysis-ready DataFrame.

In [ ]:
def prepare_titanic(raw_df, missing_age_seed=42):
    df = raw_df.copy()

    # 1. Standardize column names
    df.columns = df.columns.str.lower()
    df = df.rename(columns={
        'siblings/spouses aboard': 'sibsp',
        'parents/children aboard': 'parch',
    })

    # 2. Introduce realistic missing ages (~20%), then fill with median
    rng = np.random.default_rng(missing_age_seed)
    missing_idx = rng.choice(df.index, size=int(0.2 * len(df)), replace=False)
    df.loc[missing_idx, "age"] = np.nan
    df["age"] = df["age"].fillna(df["age"].median())

    # 3. Extract title from name, then drop the name column
    df["title"] = df["name"].str.extract(r'^(\w+\.)')
    df = df.drop(columns=["name"])

    # 4. Engineer a family feature
    df["has_family"] = ((df["sibsp"] > 0) | (df["parch"] > 0)).astype(int)

    # 5. Convert to appropriate types
    df["survived"] = df["survived"].astype("category")
    df["pclass"]   = df["pclass"].astype("category")
    df["sex"]      = df["sex"].astype("category")
    df["title"]    = df["title"].astype("category")

    return df


url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
clean = prepare_titanic(pd.read_csv(url))

print(clean.shape)
print()
print(clean.isnull().sum())
print()
clean.head()

## Your Turn

1. Load the raw Titanic CSV. Without introducing artificial missing values, demonstrate at least three non-missing-value cleaning steps on it: rename two columns, convert two columns to `category` type, and add a new engineered column of your choice.
2. Using the raw data, create a column `age_group` that is `"child"` if age < 18, `"adult"` if age is 18–60, and `"senior"` if age > 60. *(Hint: `pd.cut()` or multiple boolean assignments.)*
3. What does `pd.to_numeric(pd.Series(["7.25", "71.28", "bad", "53.10"]), errors="coerce")` return? Explain why.

In [ ]:
# Your code here